In [11]:
from keplergl import KeplerGl
import webbrowser
import os
import json
import datetime
import pandas as pd

In [12]:
date = "2026_03_04"
with open(f"../vehicle_positions_data/{date}/processed.json", "r") as f:
    raw_data = json.load(f)

In [13]:
flat_data = []

# 3. Loop through the nested hierarchy
# Step into the top-level keys (e.g., "1")
for key, route_info in raw_data.items():
    route_id = route_info.get("routeId")
    
    # Step into the list of trips
    for trip in route_info.get("trips", []):
        trip_id = trip.get("tripId")
        
        # vehicleIds is a list, so we grab the first item (if it exists)
        vehicle_ids = trip.get("vehicleIds", [])
        vehicle_id = vehicle_ids[0] if vehicle_ids else "Unknown"
        
        # Step into the actual GPS coordinates
        for pos in trip.get("vehiclePositions", []):
            
            # Build a flat dictionary for this exact moment in time
            flat_data.append({
                "route_id": route_id,
                "trip_id": trip_id,
                "vehicle_id": vehicle_id,
                "latitude": pos.get("latitude"),
                "longitude": pos.get("longitude"),
                "speed": pos.get("speed"),
                "bearing": pos.get("bearing"),
                "raw_timestamp": pos.get("timestamp")
            })

# 4. Convert the list of flat dictionaries into a Pandas DataFrame
df = pd.DataFrame(flat_data)
df.head()

,route_id,trip_id,vehicle_id,latitude,longitude,speed,bearing,raw_timestamp
0,1,2959611_0001,2864,30.244345,-97.729973,0.0,0.0,1772619444
1,1,2959611_0001,2864,30.244347,-97.729965,0.0,0.0,1772619459
2,1,2959611_0001,2864,30.244347,-97.729965,0.0,0.0,1772619474
3,1,2959611_0001,2864,30.244347,-97.729973,0.0,0.0,1772619492
4,1,2959611_0001,2864,30.244347,-97.729973,0.0,0.0,1772619504


In [2]:
# Austin, TX coordinates
AUSTIN_LAT = 30.2672
AUSTIN_LON = -97.7431

# Kepler.gl uses a JSON-like dictionary for configuration.
# Here we set the initial view (mapState) to center on Austin.
map_config = {
    "version": "v1",
    "config": {
        "mapState": {
            "latitude": AUSTIN_LAT,
            "longitude": AUSTIN_LON,
            "zoom": 11,     # Adjust zoom level (higher is closer)
            "pitch": 0,     # Adjust for 3D tilt (0 to 60)
            "bearing": 0    # Map rotation
        }
    }
}

map_config

{'version': 'v1',
 'config': {'mapState': {'latitude': 30.2672,
   'longitude': -97.7431,
   'zoom': 11,
   'pitch': 0,
   'bearing': 0}}}

In [3]:
# Initialize the Kepler.gl map object
# The height parameter controls the size of the widget if you run this in a Jupyter Notebook
austin_map = KeplerGl(height=600, config=map_config)

file_name = 'austin_base_map.html'
austin_map.save_to_html(file_name=file_name)

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter
Map saved to austin_base_map.html!


In [4]:
file_path = 'file://' + os.path.realpath(file_name)
webbrowser.open_new_tab(file_path)

True